# Notebook 00: The SimASM Pipeline

This notebook demonstrates how **simasm** works: from event graph specification to executable simulation to complexity analysis.

SimASM is an Abstract State Machine (ASM) framework for discrete-event simulation. It provides:
1. A **formal language** for expressing simulation models
2. An **automatic converter** from event graph JSON specs to SimASM code
3. A **native runtime** that executes SimASM models directly
4. A **complexity analyzer** (HET) that measures structural model complexity

We walk through a single model — the Tandem 1-Queue — to show the full pipeline.

In [ ]:
# Install simasm (uncomment in Colab)
# !pip install simasm==0.4.8

import simasm
print(f"SimASM version: {simasm.__version__}")

## Step 1: The Event Graph Specification

Each model starts as a **JSON event graph** — the algebraic specification
$S = (F, C, T, \Gamma, G)$ from Schruben (1983):
- **Vertices** = state transition functions (events)
- **Edges** = scheduling relationships with conditions and delays
- **Random streams** = stochastic delay distributions

In [ ]:
# Tandem 1-Queue Event Graph JSON
tandem_1_eg_json = {
  "model_name": "tandem_1_eg",
  "description": "Tandem 1-Queue using Event Graph formalism - 1 station in series",

  "entities": {
    "Load": {
      "name": "Load",
      "parent": "Object",
      "attributes": {
        "arrival_time": "Real",
        "service_start_time_1": "Real",
        "departure_time": "Real"
      }
    }
  },

  "parameters": {
    "service_capacity": {"type": "Nat", "value": 5, "description": "Number of servers per station"},
    "iat_mean": {"type": "Real", "value": 1.25, "description": "Inter-arrival time mean"},
    "ist_mean": {"type": "Real", "value": 1.0, "description": "Service time mean per station"},
    "sim_end_time": {"type": "Real", "value": 10000.0, "description": "Simulation end time"}
  },

  "state_variables": {
    "queue_count_1": {"type": "Nat", "initial": 0, "description": "Number in queue at station 1"},
    "server_count_1": {"type": "Nat", "initial": 0, "description": "Number being served at station 1"},
    "load_id_counter": {"type": "Nat", "initial": 0},
    "departure_count": {"type": "Nat", "initial": 0}
  },

  "random_streams": {
    "interarrival_time": {
      "distribution": "exponential",
      "params": {"mean": "iat_mean"},
      "stream_name": "arrivals"
    },
    "service_time_1": {
      "distribution": "exponential",
      "params": {"mean": "ist_mean"},
      "stream_name": "service_1"
    }
  },

  "vertices": [
    {
      "name": "Arrive",
      "description": "Customer arrival event",
      "state_change": "load_id_counter := load_id_counter + 1; queue_count_1 := queue_count_1 + 1"
    },
    {
      "name": "Start_1",
      "parameters": [{"load": "Load"}],
      "description": "Service start at station 1",
      "state_change": "queue_count_1 := queue_count_1 - 1; server_count_1 := server_count_1 + 1"
    },
    {
      "name": "Finish_1",
      "parameters": [{"load": "Load"}],
      "description": "Service completion at station 1 (departure from system)",
      "state_change": "server_count_1 := server_count_1 - 1; departure_count := departure_count + 1"
    }
  ],

  "scheduling_edges": [
    {"from": "Arrive", "to": "Arrive", "delay": "interarrival_time", "condition": "true", "priority": 0},
    {"from": "Arrive", "to": "Start_1", "delay": 0, "condition": "server_count_1 < service_capacity", "priority": 0, "parameters": ["load"]},
    {"from": "Start_1", "to": "Finish_1", "delay": "service_time_1", "condition": "true", "priority": 0, "parameters": ["load"]},
    {"from": "Finish_1", "to": "Start_1", "delay": 0, "condition": "queue_count_1 > 0 and server_count_1 < service_capacity", "priority": 0, "parameters": ["next_load"]}
  ],

  "cancelling_edges": [],

  "initial_events": [{"event": "Arrive", "time": "interarrival_time"}],

  "stopping_condition": "sim_clocktime >= sim_end_time",

  "observables": {
    "queue_count_1": {"name": "queue_count_1", "expression": "queue_count_1", "description": "Number in queue at station 1"},
    "server_count_1": {"name": "server_count_1", "expression": "server_count_1", "description": "Number of busy servers at station 1"},
    "in_system": {"name": "in_system", "expression": "queue_count_1 + server_count_1", "description": "Total in system"},
    "L_q_total": {"name": "L_q_total", "expression": "queue_count_1", "description": "Total queue length"},
    "departure_count": {"name": "departure_count", "expression": "departure_count", "description": "Total departures"}
  },

  "statistics": [
    {"name": "L_q_total", "type": "time_average", "observable": "L_q_total", "description": "Average total queue length"},
    {"name": "L", "type": "time_average", "observable": "in_system", "description": "Average number in system"},
    {"name": "throughput", "type": "count", "observable": "departure_count", "description": "Total departures"}
  ]
}

# Save JSON to file for the converter
import json
with open("tandem_1_eg.json", "w") as f:
    json.dump(tandem_1_eg_json, f, indent=2)
print("Saved tandem_1_eg.json")

## Step 2: Convert to SimASM Assembly

The `%%simasm convert` magic translates the JSON event graph into SimASM assembly code.
The generated code is a formal ASM program implementing the **Next-Event Time-Advance Algorithm** (Law 2013):
1. **Initialization**: Set clock=0, STATE=STATE_0, FEL=FEL_0
2. **Timing**: Pop earliest event from FEL, advance clock
3. **Event routine**: Execute $f_v$, process scheduling edges with $c_e$ and $t_e$
4. Repeat until termination condition

In [ ]:
%%simasm convert

convert tandem_1_eg:
    source: "tandem_1_eg.json"
    formalism: event_graph
    register: "tandem_1_eg"
    print: true
endconvert

### The SimASM Language

The generated code is a **complete formal program**, not pseudocode.
Each event is a `rule` that updates state variables and schedules future events.

Key observations:
- State updates are **explicit**: `queue_count_1 := queue_count_1 + 1`
- Event scheduling is **formal**: create event object, set time, add to FEL
- Conditions map directly to event graph edges: `if server_count_1 < service_capacity`
- Random streams sample automatically: `sim_clocktime + interarrival_time`

The assembly code captures the **full operational semantics** — every state update,
every conditional branch, every scheduling decision. This is what makes
SimASM complexity (SMC) a more informative metric than LOC or cyclomatic complexity.

## Step 3: Run the Model

SimASM can execute models natively through its experiment runner.
The `%%simasm experiment` magic defines replications, warm-up periods, and output statistics.

In [ ]:
%%simasm experiment
// Quick experiment: 3 replications, 1000h each

experiment Tandem1Test:
    model := "tandem_1_eg"

    replication:
        count: 3
        warm_up_time: 100.0
        run_length: 1000.0
        seed_strategy: "incremental"
        base_seed: 42
    endreplication

    statistics:
        stat L: time_average
            expression: "in_system()"
        endstat
        stat L_q: time_average
            expression: "L_q_total()"
        endstat
        stat throughput: count
            expression: "departure_count()"
        endstat
    endstatistics
endexperiment

## Step 4: Analyze Complexity

The `%%simasm complexity` magic computes the **Hierarchical Execution Time (HET)** of each rule
by counting microsteps: state updates, conditionals, let-bindings, function calls,
entity creations, and list operations.

The **SimASM Model Complexity (SMC)** is computed from the deduplicated entity traversal paths
through the event graph, plus the control overhead (Nowack 2000, Definition 4.3).

In [ ]:
%%simasm complexity
// HET analysis for the Tandem 1-Queue model

complexity Tandem1Complexity:
    models:
        model tandem_1_eg:
            simasm: "tandem_1_eg"
            event_graph: "tandem_1_eg.json"
        endmodel
    endmodels
    metrics:
        het_static: true
        het_path_based: true
        structural: true
        component_breakdown: true
    endmetrics
endcomplexity

## Step 5: Why SMC Matters

Compare complexity metrics for tandem models of increasing size.
As stations are added, each event rule must manage more conditional branches
and state updates — SMC captures this combinatorial explosion.

In [ ]:
%%simasm complexity
// Scaling analysis: tandem models of increasing size

complexity TandemScaling:
    models:
        model tandem_1:
            simasm: "../data/simasm/tandem/tandem_1_eg.simasm"
            event_graph: "../data/models/tandem/tandem_1_eg.json"
        endmodel
        model tandem_3:
            simasm: "../data/simasm/tandem/tandem_3_eg.simasm"
            event_graph: "../data/models/tandem/tandem_3_eg.json"
        endmodel
        model tandem_5:
            simasm: "../data/simasm/tandem/tandem_5_eg.simasm"
            event_graph: "../data/models/tandem/tandem_5_eg.json"
        endmodel
        model tandem_10:
            simasm: "../data/simasm/tandem/tandem_10_eg.simasm"
            event_graph: "../data/models/tandem/tandem_10_eg.json"
        endmodel
        model tandem_20:
            simasm: "../data/simasm/tandem/tandem_20_eg.simasm"
            event_graph: "../data/models/tandem/tandem_20_eg.json"
        endmodel
    endmodels
    metrics:
        het_static: true
        het_path_based: true
        structural: true
        component_breakdown: true
    endmetrics
endcomplexity

## Summary

The SimASM pipeline uses four `%%simasm` magics:

```
Event Graph (JSON)  ──%%simasm convert──>  SimASM Assembly (.simasm)
                                                  │
                                   ┌──────────────┼──────────────┐
                                   │              │              │
                          %%simasm experiment  %%simasm complexity  register
                                   │              │              │
                            Simulation       SMC / HET      Model Registry
                            Results          Metrics        (reuse in expts)
```

The key insight: SimASM's assembly code is a **formal intermediate representation**
that sits between the high-level event graph specification and the executable simulation.
Its structured complexity (SMC) captures model difficulty that surface-level metrics
(LOC, cyclomatic number) miss.

### References

- Schruben, L.W. (1983). Simulation Modeling with Event Graphs. Communications of the ACM.
- Law, A.M. (2013). Simulation Modeling and Analysis (5th ed.). McGraw-Hill.
- Nowack, A. (2000). Modular Algebraic Specification of Discrete Event Systems. Thesis.